# SentinelOps Arena — GRPO Training with Unsloth

This notebook demonstrates how to train the **Worker Agent** using GRPO (Group Relative Policy Optimization) on the SentinelOps Arena environment.

SentinelOps Arena is a multi-agent self-play RL environment for enterprise security training built on OpenEnv. We are targeting the **Fleet AI (Scalable Oversight)** and **Patronus AI (Schema Drift)** tracks.

## 1. Setup Environment

In [ ]:
!pip install "openenv-core[core]>=0.2.0" mcp fastmcp pydantic pandas
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import os
if not os.path.exists("NexusEnv"):
    !git clone https://github.com/nihalnihalani/NexusEnv.git
import sys
sys.path.append("/content/NexusEnv")

## 2. Collect Training Data via Self-Play

We run the environment using our heuristic agents to generate the initial "prompts" that the Worker agent will face during training.

In [ ]:
import json
from datasets import Dataset
from NexusEnv.train import build_training_dataset, WORKER_SYSTEM_PROMPT

NUM_EPISODES = 5
print(f"Collecting training data from {NUM_EPISODES} episodes...")
dataset_raw = build_training_dataset(num_episodes=NUM_EPISODES, target_agent="worker")

prompts = []
for d in dataset_raw:
    messages = [
        {"role": "system", "content": WORKER_SYSTEM_PROMPT},
        {"role": "user", "content": d["prompt"]},
    ]
    prompts.append(messages)

train_dataset = Dataset.from_dict({"prompt": prompts})
print(f"Dataset generated with {len(train_dataset)} examples.")

## 3. Load Model with Unsloth

We use `Qwen/Qwen2.5-0.5B-Instruct` as it fits comfortably in a free Colab T4 GPU.

In [ ]:
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True, # Enable vLLM fast inference
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

## 4. GRPO Training

We set up the GRPO configuration and launch the training process.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from NexusEnv.train import make_reward_function

reward_fn = make_reward_function("worker")

grpo_config = GRPOConfig(
    output_dir="./sentinelops-grpo-worker",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    learning_rate=5e-6,
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
)

trainer.train()

## 5. Save the Trained Model

Finally, we save our GRPO-trained LoRA weights.

In [ ]:
output_dir = "./sentinelops-grpo-worker"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Model saved successfully!")